Step 1: Prepare the recorded aedat4.

- convert to h5 for numpy (some operations were not possible with using the aedat4 directly)
- cut video to about the start of movement

In [1]:
from dv import AedatFile
import numpy as np
import h5py
import time

INPUT = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave-2026_07_09_18_41_14.aedat4"
OUT   = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_split_cut3s.h5"

CUT_SECONDS = 4.0
CUT_US = int(CUT_SECONDS * 1_000_000)  # timestamps in microseconds

CHUNK_EVENTS = 1_000_000   # flush buffer every N events per polarity (tune)
PRINT_EVERY_S = 1.0        # print progress every 1 second of recording time

def append_rows(dset, rows_np):
    """Append rows to a resizable HDF5 dataset."""
    if rows_np.size == 0:
        return
    n_old = dset.shape[0]
    n_new = n_old + rows_np.shape[0]
    dset.resize((n_new, 3))
    dset[n_old:n_new, :] = rows_np

with h5py.File(OUT, "w") as h5:
    dpos = h5.create_dataset(
        "pos", shape=(0, 3), maxshape=(None, 3),
        dtype=np.int64, chunks=(200_000, 3), compression="gzip"
    )
    dneg = h5.create_dataset(
        "neg", shape=(0, 3), maxshape=(None, 3),
        dtype=np.int64, chunks=(200_000, 3), compression="gzip"
    )

    buf_pos = []
    buf_neg = []

    with AedatFile(INPUT) as f:
        print(f.names)

        t0 = None
        last_print_t = None
        wall_start = time.time()

        kept_pos = 0
        kept_neg = 0

        for e in f["events"]:
            t = int(e.timestamp)

            if t0 is None:
                t0 = t
                last_print_t = t0

            # skip first 4 seconds
            if t < t0 + CUT_US:
                continue

            if e.polarity:
                buf_pos.append((e.x, e.y, t))
                if len(buf_pos) >= CHUNK_EVENTS:
                    append_rows(dpos, np.asarray(buf_pos, dtype=np.int64))
                    kept_pos += len(buf_pos)
                    buf_pos.clear()
            else:
                buf_neg.append((e.x, e.y, t))
                if len(buf_neg) >= CHUNK_EVENTS:
                    append_rows(dneg, np.asarray(buf_neg, dtype=np.int64))
                    kept_neg += len(buf_neg)
                    buf_neg.clear()

            # progress print based on recording time
            if (t - last_print_t) >= int(PRINT_EVERY_S * 1_000_000):
                rec_s = (t - t0) / 1_000_000
                wall_s = time.time() - wall_start
                rate = (kept_pos + kept_neg) / wall_s if wall_s > 0 else 0.0
                print(f"processed {rec_s:7.2f} s | kept pos={kept_pos:,} neg={kept_neg:,} | {rate:,.0f} ev/s")
                last_print_t = t

        # flush remaining buffers
        if buf_pos:
            append_rows(dpos, np.asarray(buf_pos, dtype=np.int64))
            kept_pos += len(buf_pos)
            buf_pos.clear()

        if buf_neg:
            append_rows(dneg, np.asarray(buf_neg, dtype=np.int64))
            kept_neg += len(buf_neg)
            buf_neg.clear()

    h5.attrs["t0"] = t0
    h5.attrs["cut_us"] = CUT_US

print("saved:", OUT)


['events', 'imu', 'triggers']
processed    4.00 s | kept pos=0 neg=0 | 0 ev/s
processed    5.00 s | kept pos=0 neg=0 | 0 ev/s
processed    6.00 s | kept pos=0 neg=0 | 0 ev/s
processed    7.00 s | kept pos=9,000,000 neg=9,000,000 | 133,704 ev/s
processed    8.00 s | kept pos=19,000,000 neg=21,000,000 | 135,538 ev/s
processed    9.00 s | kept pos=30,000,000 neg=33,000,000 | 138,222 ev/s
processed   10.00 s | kept pos=41,000,000 neg=44,000,000 | 139,565 ev/s
processed   11.00 s | kept pos=52,000,000 neg=56,000,000 | 140,590 ev/s
processed   12.00 s | kept pos=54,000,000 neg=58,000,000 | 139,833 ev/s
processed   13.00 s | kept pos=54,000,000 neg=58,000,000 | 139,685 ev/s
processed   14.00 s | kept pos=54,000,000 neg=58,000,000 | 139,542 ev/s
processed   15.00 s | kept pos=54,000,000 neg=58,000,000 | 139,393 ev/s
processed   16.00 s | kept pos=55,000,000 neg=58,000,000 | 140,405 ev/s
processed   17.00 s | kept pos=55,000,000 neg=58,000,000 | 140,241 ev/s
processed   18.00 s | kept pos=55,00

Step 2: Match time by looking for spike increase

In [2]:
import h5py
import numpy as np

IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_split_cut3s.h5"

BIN_US = 500         # 1 ms bins (try 500 or 2000 too)
THRESH_Z = 4.0         # spike threshold in standard deviations
BASELINE_S = 1       # use first 1s to estimate baseline

with h5py.File(IN_H5, "r") as h5:
    dpos = h5["pos"]
    dneg = h5["neg"]

    tpos = dpos[:, 2] if dpos.shape[0] else np.array([], dtype=np.int64)
    tneg = dneg[:, 2] if dneg.shape[0] else np.array([], dtype=np.int64)

# merge timestamps only (cheap)
t_all = np.sort(np.concatenate([tpos, tneg]))
t0 = int(t_all[0])
t1 = int(t_all[-1])

# bin index for each event
bins = (t_all - t0) // BIN_US
nbins = int(bins[-1]) + 1

counts = np.bincount(bins.astype(np.int64), minlength=nbins)

# baseline from first BASELINE_S seconds
baseline_bins = int((BASELINE_S * 1_000_000) // BIN_US)
mu = counts[:baseline_bins].mean()
sigma = counts[:baseline_bins].std() + 1e-9

z = (counts - mu) / sigma
spike_bins = np.where(z > THRESH_Z)[0]

print(f"baseline mean={mu:.1f} ev/bin, std={sigma:.1f}, spikes found={len(spike_bins)}")

# print first few spike times
for b in spike_bins[:100]:
    t_spike = t0 + b * BIN_US
    print("spike at t =", t_spike, "count =", counts[b], "z =", z[b])


baseline mean=67.4 ev/bin, std=8.1, spikes found=15297
spike at t = 1783615280735544 count = 104 z = 4.496899841742876
spike at t = 1783615280736044 count = 297 z = 28.2402259105564
spike at t = 1783615280736544 count = 234 z = 20.489813774104316
spike at t = 1783615280737044 count = 285 z = 26.763956932184573
spike at t = 1783615280737544 count = 235 z = 20.612836188968636
spike at t = 1783615280738044 count = 539 z = 58.011650307721546
spike at t = 1783615280738544 count = 702 z = 78.0643039306055
spike at t = 1783615280739044 count = 1335 z = 155.9374925397193
spike at t = 1783615280739544 count = 1020 z = 117.18543185745888
spike at t = 1783615280740044 count = 1606 z = 189.2765669679497
spike at t = 1783615280740544 count = 1372 z = 160.48932188969908
spike at t = 1783615280741044 count = 2436 z = 291.38517130533427
spike at t = 1783615280741544 count = 1645 z = 194.07444114765812
spike at t = 1783615280742044 count = 3149 z = 379.1001531035936
spike at t = 1783615280742544 count 

Step 3: Cut the h5 at the spike increase

In [3]:
import h5py
import numpy as np

def cut_h5_at_timestamp(IN_H5, OUT_H5, t_start_us):
    """
    Keep events with timestamp >= t_start_us and shift time so that t_start_us -> 0.
    Assumes datasets:
        pos: (N,3) [x,y,t]
        neg: (M,3)
    and timestamps are sorted.
    """

    with h5py.File(IN_H5, "r") as h5in, h5py.File(OUT_H5, "w") as h5out:

        for name in ["pos", "neg"]:
            if name not in h5in:
                continue

            d = h5in[name]
            print(f"Processing {name}, shape={d.shape}")

            if d.shape[0] == 0:
                h5out.create_dataset(name, data=d[:], dtype=d.dtype)
                continue

            # timestamps column
            t = d[:, 2]

            # find first index where t >= t_start_us
            i0 = int(np.searchsorted(t, t_start_us, side="left"))

            print(f"{name}: keeping {d.shape[0] - i0} events")

            if i0 >= d.shape[0]:
                # nothing left
                h5out.create_dataset(name, shape=(0, 3), dtype=np.int32)
                continue

            # slice remaining events
            ev = d[i0:, :].astype(np.int32)

            # shift timestamps so t_start_us -> 0
            ev[:, 2] -= int(t_start_us)

            h5out.create_dataset(name, data=ev, dtype=np.int32, chunks=True)

    print("Saved trimmed file:", OUT_H5)
    
IN_H5  = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_split_cut3s.h5"
OUT_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_sync.h5"

# Example: start at 6.2 seconds
t_start_us = 1783615280736044

cut_h5_at_timestamp(IN_H5, OUT_H5, t_start_us)


Processing pos, shape=(55801320, 3)
pos: keeping 55641805 events
Processing neg, shape=(58957671, 3)
neg: keeping 58833204 events
Saved trimmed file: /home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_sync.h5


Step 4: Calibrating PROJECTION_W_PX

In [5]:
#!/usr/bin/env python3
import h5py
import numpy as np
import cv2

# ============================
# PATHS
# ============================
IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_test_sync.h5"
ANGLE_CSV = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/angle_log.csv"

# ============================
# CAMERA / PANORAMA
# ============================
W, H = 640, 480
PAN_H = 800
PAN_MARGIN_PX = 800

DIRECTION = "right_to_left"
ROTATION_SIGN = 1.0
ROTATION_FACTOR = 1.0
TIME_OFFSET_US = 0

# ============================
# CALIBRATION SEARCH
# ============================
TEST_DURATION_US = 500_000   # use first 0.5 seconds
WINDOW_US = 1000

COARSE_MIN = 3000
COARSE_MAX = 15000
COARSE_STEP = 500

FINE_RADIUS = 300
FINE_STEP = 50

ACC_GAIN = 1.0
DECIMATION_KEEP = 0.25
RNG_SEED = 123
# ============================


def load_angle_csv(path):
    arr = np.loadtxt(path, delimiter=",", skiprows=1)

    t = np.round(arr[:, 0] * 1_000_000).astype(np.int64)
    a = arr[:, 1].astype(np.float64)

    nz = np.where(a != 0)[0]
    if len(nz):
        t = t[nz[0]:]
        a = a[nz[0]:]

    t -= t[0]
    a_unwrapped = np.rad2deg(np.unwrap(np.deg2rad(a)))

    return t, a_unwrapped


def interp_angle(t_query, t_angle, angle_unwrapped):
    tq = t_query.astype(np.float64) + TIME_OFFSET_US
    a = np.interp(tq, t_angle.astype(np.float64), angle_unwrapped)
    return np.mod(a, 360.0)


def signed_angle_around_zero(a):
    return ((a + 180.0) % 360.0) - 180.0


def angle_to_x(angle_deg, projection_w_px, pan_w):
    a = signed_angle_around_zero(angle_deg)

    if DIRECTION == "right_to_left":
        a = -a

    return pan_w / 2.0 + (a / 360.0) * projection_w_px


def decimate_rows(rows, keep, rng):
    if rows.shape[0] == 0 or keep >= 1.0:
        return rows

    mask = rng.random(rows.shape[0]) < keep
    return rows[mask]


def correct_xy_for_projection(ev, t_sync, t_angle, angle_unwrapped, projection_w_px, pan_w):
    if ev.shape[0] == 0:
        return np.zeros((0, 2), dtype=np.int64)

    xy = ev[:, :2].astype(np.float64)
    t = t_sync.astype(np.int64)

    a = interp_angle(t, t_angle, angle_unwrapped)

    x0 = xy[:, 0] - (W - 1) / 2.0
    y0 = xy[:, 1] - (H - 1) / 2.0

    theta = ROTATION_SIGN * ROTATION_FACTOR * np.deg2rad(a)

    c = np.cos(theta)
    s = np.sin(theta)

    xr = x0 * c - y0 * s
    yr = x0 * s + y0 * c

    pano_x = angle_to_x(a, projection_w_px, pan_w) + xr
    pano_y = PAN_H / 2.0 + yr

    valid = (
        (pano_x >= 0) & (pano_x < pan_w) &
        (pano_y >= 0) & (pano_y < PAN_H)
    )

    return np.column_stack([
        np.round(pano_x[valid]).astype(np.int64),
        np.round(pano_y[valid]).astype(np.int64)
    ])


def accumulate_xy(acc, xy):
    if xy.shape[0] == 0:
        return

    x = xy[:, 0]
    y = xy[:, 1]

    valid = (
        (x >= 0) & (x < acc.shape[1]) &
        (y >= 0) & (y < acc.shape[0])
    )

    x = x[valid]
    y = y[valid]

    if x.size == 0:
        return

    idx = y * acc.shape[1] + x
    counts = np.bincount(idx, minlength=acc.size).reshape(acc.shape)
    acc += counts.astype(np.float32) * ACC_GAIN


def sharpness_score(acc):
    """
    Higher score = sharper / less smeared.
    Sobel-x rewards sharp vertical structures, which is useful for horizontal smear.
    """
    if acc.max() <= 0:
        return -1.0

    img = acc.astype(np.float32)

    # normalize to avoid score being dominated by event count
    img = img / (img.max() + 1e-12)

    sobel_x = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)

    # ignore empty background
    mask = img > 0.01
    if np.count_nonzero(mask) < 100:
        return -1.0

    return float(np.var(sobel_x[mask]))


def score_projection_width(
    projection_w_px,
    h5,
    tpos,
    tneg,
    t_angle,
    angle_unwrapped,
    rng
):
    pan_w = int(projection_w_px + 2 * PAN_MARGIN_PX)
    acc = np.zeros((PAN_H, pan_w), dtype=np.float32)

    dpos = h5["pos"]
    dneg = h5["neg"]

    t_max = min(max(tpos[-1], tneg[-1]), TEST_DURATION_US)
    windows = np.arange(0, t_max, WINDOW_US, dtype=np.int64)

    for t_start in windows:
        t_end = t_start + WINDOW_US

        pi0 = np.searchsorted(tpos, t_start, side="left")
        pi1 = np.searchsorted(tpos, t_end, side="left")

        ni0 = np.searchsorted(tneg, t_start, side="left")
        ni1 = np.searchsorted(tneg, t_end, side="left")

        pos = np.asarray(dpos[pi0:pi1, :], dtype=np.int64)
        neg = np.asarray(dneg[ni0:ni1, :], dtype=np.int64)

        pos = decimate_rows(pos, DECIMATION_KEEP, rng)
        neg = decimate_rows(neg, DECIMATION_KEEP, rng)

        pos_xy = correct_xy_for_projection(
            pos,
            tpos[pi0:pi0 + pos.shape[0]] if pos.shape[0] else np.array([], dtype=np.int64),
            t_angle,
            angle_unwrapped,
            projection_w_px,
            pan_w
        )

        neg_xy = correct_xy_for_projection(
            neg,
            tneg[ni0:ni0 + neg.shape[0]] if neg.shape[0] else np.array([], dtype=np.int64),
            t_angle,
            angle_unwrapped,
            projection_w_px,
            pan_w
        )

        accumulate_xy(acc, pos_xy)
        accumulate_xy(acc, neg_xy)

    return sharpness_score(acc)


def run_search(values, h5, tpos, tneg, t_angle, angle_unwrapped, label):
    best_val = None
    best_score = -1.0
    results = []

    print(f"\n{label} search:")

    for v in values:
        rng = np.random.default_rng(RNG_SEED)

        score = score_projection_width(
            int(v),
            h5,
            tpos,
            tneg,
            t_angle,
            angle_unwrapped,
            rng
        )

        results.append((int(v), score))

        print(f"PROJECTION_W_PX={int(v):5d}  score={score:.8f}")

        if score > best_score:
            best_score = score
            best_val = int(v)

    print(f"{label} best:", best_val, "score:", best_score)
    return best_val, best_score, results


def main():
    t_angle, angle_unwrapped = load_angle_csv(ANGLE_CSV)

    with h5py.File(IN_H5, "r") as h5:
        dpos = h5["pos"]
        dneg = h5["neg"]

        tpos_raw = dpos[:, 2].astype(np.int64)
        tneg_raw = dneg[:, 2].astype(np.int64)

        t0 = min(tpos_raw[0], tneg_raw[0])
        tpos = tpos_raw - t0
        tneg = tneg_raw - t0

        coarse_values = np.arange(COARSE_MIN, COARSE_MAX + 1, COARSE_STEP)

        coarse_best, _, coarse_results = run_search(
            coarse_values,
            h5,
            tpos,
            tneg,
            t_angle,
            angle_unwrapped,
            "Coarse"
        )

        fine_min = max(500, coarse_best - FINE_RADIUS)
        fine_max = coarse_best + FINE_RADIUS
        fine_values = np.arange(fine_min, fine_max + 1, FINE_STEP)

        fine_best, fine_score, fine_results = run_search(
            fine_values,
            h5,
            tpos,
            tneg,
            t_angle,
            angle_unwrapped,
            "Fine"
        )

    print("\n====================================")
    print("Recommended PROJECTION_W_PX:", fine_best)
    print("Best score:", fine_score)
    print("Use PAN_W:", fine_best + 2 * PAN_MARGIN_PX)
    print("PAN_MARGIN_PX:", PAN_MARGIN_PX)
    print("====================================")


if __name__ == "__main__":
    main()

KeyboardInterrupt: 

Step 5: calibrate timing offset

In [10]:
#!/usr/bin/env python3
import h5py
import numpy as np
import cv2

RAW_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/dvSave_test_sync.h5"
ANGLE_CSV = "/home/maurice/Documents/Master_Data/Periscope/Static/angle_log.csv"

W, H = 640, 480

PROJECTION_W_PX = 11500
PAN_MARGIN_PX = 800
PAN_W = PROJECTION_W_PX + 2 * PAN_MARGIN_PX
PAN_H = 800

DIRECTION = "right_to_left"
ROTATION_SIGN = 1.0
ROTATION_FACTOR = 1.0
ANGLE_OFFSET_DEG = 0

TEST_DURATION_US = 1_000_000

OFFSET_MIN_US = -30_000
OFFSET_MAX_US = 5_000
OFFSET_STEP_US = 100

DECIMATION_KEEP = 0.25
RNG_SEED = 123

LOG_ACC_MIN = 1.5


def load_angle_csv(path):
    arr = np.loadtxt(path, delimiter=",", skiprows=1)

    t = np.round(arr[:, 0] * 1_000_000).astype(np.int64)
    a = arr[:, 1].astype(np.float64)

    nz = np.where(a != 0)[0]
    if len(nz):
        t = t[nz[0]:]
        a = a[nz[0]:]

    t -= t[0]
    a_unwrapped = np.rad2deg(np.unwrap(np.deg2rad(a)))

    return t, a_unwrapped


def signed_angle_around_zero(a):
    return ((a + 180.0) % 360.0) - 180.0


def angle_to_x(angle_deg):
    a = signed_angle_around_zero(angle_deg)

    if DIRECTION == "right_to_left":
        a = -a

    return PAN_W / 2.0 + (a / 360.0) * PROJECTION_W_PX


def interp_angle(t_query, t_angle, angle_unwrapped, offset_us):
    tq = t_query.astype(np.float64) + offset_us

    a = np.interp(
        tq,
        t_angle.astype(np.float64),
        angle_unwrapped.astype(np.float64)
    )

    a += ANGLE_OFFSET_DEG
    return np.mod(a, 360.0)


def correct_events(ev, t_sync, t_angle, angle_unwrapped, offset_us):
    if ev.shape[0] == 0:
        return np.zeros((0, 2), dtype=np.int64)

    xy = ev[:, :2].astype(np.float64)
    t = t_sync.astype(np.int64)

    a = interp_angle(t, t_angle, angle_unwrapped, offset_us)

    x0 = xy[:, 0] - (W - 1) / 2.0
    y0 = xy[:, 1] - (H - 1) / 2.0

    theta = ROTATION_SIGN * ROTATION_FACTOR * np.deg2rad(a)

    c = np.cos(theta)
    s = np.sin(theta)

    xr = x0 * c - y0 * s
    yr = x0 * s + y0 * c

    x = angle_to_x(a) + xr
    y = PAN_H / 2.0 + yr

    valid = (
        (x >= 0) & (x < PAN_W) &
        (y >= 0) & (y < PAN_H)
    )

    return np.column_stack([
        np.round(x[valid]).astype(np.int64),
        np.round(y[valid]).astype(np.int64)
    ])


def accumulate_xy(xy):
    acc = np.zeros((PAN_H, PAN_W), dtype=np.float32)

    if xy.shape[0] == 0:
        return acc

    x = xy[:, 0]
    y = xy[:, 1]

    valid = (
        (x >= 0) & (x < PAN_W) &
        (y >= 0) & (y < PAN_H)
    )

    x = x[valid]
    y = y[valid]

    if x.size == 0:
        return acc

    idx = y * PAN_W + x
    acc += np.bincount(idx, minlength=PAN_H * PAN_W).reshape(PAN_H, PAN_W)

    return acc


def sharpness_score(acc):
    img = np.log1p(acc)

    img[img < LOG_ACC_MIN] = 0.0

    if np.count_nonzero(img) < 100:
        return -1.0

    img /= img.max() + 1e-12

    sobel_x = cv2.Sobel(img.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)

    mask = img > 0

    return float(np.var(sobel_x[mask]))


def decimate(ev, t, keep, rng):
    if ev.shape[0] == 0 or keep >= 1.0:
        return ev, t

    mask = rng.random(ev.shape[0]) < keep
    return ev[mask], t[mask]


def main():
    t_angle, angle_unwrapped = load_angle_csv(ANGLE_CSV)

    with h5py.File(RAW_H5, "r") as h5:
        dpos = h5["pos"]
        dneg = h5["neg"]

        tpos_raw = dpos[:, 2].astype(np.int64)
        tneg_raw = dneg[:, 2].astype(np.int64)

        t0 = min(tpos_raw[0], tneg_raw[0])

        tpos = tpos_raw - t0
        tneg = tneg_raw - t0

        pi1 = np.searchsorted(tpos, TEST_DURATION_US, side="left")
        ni1 = np.searchsorted(tneg, TEST_DURATION_US, side="left")

        pos = np.asarray(dpos[:pi1, :], dtype=np.int64)
        neg = np.asarray(dneg[:ni1, :], dtype=np.int64)

        pos_t = tpos[:pi1]
        neg_t = tneg[:ni1]

    rng = np.random.default_rng(RNG_SEED)

    pos, pos_t = decimate(pos, pos_t, DECIMATION_KEEP, rng)
    neg, neg_t = decimate(neg, neg_t, DECIMATION_KEEP, rng)

    print("Testing first second")
    print("pos events:", pos.shape[0])
    print("neg events:", neg.shape[0])
    print("offset range:", OFFSET_MIN_US, "to", OFFSET_MAX_US)

    best_offset = None
    best_score = -1.0

    for offset in range(OFFSET_MIN_US, OFFSET_MAX_US + 1, OFFSET_STEP_US):
        pos_xy = correct_events(pos, pos_t, t_angle, angle_unwrapped, offset)
        neg_xy = correct_events(neg, neg_t, t_angle, angle_unwrapped, offset)

        xy = np.vstack([pos_xy, neg_xy]) if pos_xy.shape[0] or neg_xy.shape[0] else np.zeros((0, 2), dtype=np.int64)

        acc = accumulate_xy(xy)
        score = sharpness_score(acc)

        print(f"TIME_OFFSET_US={offset:+7d} score={score:.8f}")

        if score > best_score:
            best_score = score
            best_offset = offset

    print("\n====================================")
    print("BEST TIME_OFFSET_US =", best_offset)
    print("BEST SCORE =", best_score)
    print("Use this in your correction script:")
    print(f"TIME_OFFSET_US = {best_offset}")
    print("====================================")


if __name__ == "__main__":
    main()

Testing first second
pos events: 3053727
neg events: 3498809
offset range: -30000 to 5000
TIME_OFFSET_US= -30000 score=0.78711665
TIME_OFFSET_US= -29900 score=0.76430148
TIME_OFFSET_US= -29800 score=0.76348990
TIME_OFFSET_US= -29700 score=0.73974407
TIME_OFFSET_US= -29600 score=0.81908286
TIME_OFFSET_US= -29500 score=0.78913742
TIME_OFFSET_US= -29400 score=0.76521015
TIME_OFFSET_US= -29300 score=0.81686515
TIME_OFFSET_US= -29200 score=0.79048288
TIME_OFFSET_US= -29100 score=0.81741440
TIME_OFFSET_US= -29000 score=0.74720955
TIME_OFFSET_US= -28900 score=0.78931850
TIME_OFFSET_US= -28800 score=0.76646256
TIME_OFFSET_US= -28700 score=0.76847529
TIME_OFFSET_US= -28600 score=0.81892085
TIME_OFFSET_US= -28500 score=0.74453682
TIME_OFFSET_US= -28400 score=0.79254961
TIME_OFFSET_US= -28300 score=0.74847597
TIME_OFFSET_US= -28200 score=0.76986420
TIME_OFFSET_US= -28100 score=0.79431522
TIME_OFFSET_US= -28000 score=0.73015612
TIME_OFFSET_US= -27900 score=0.77487165
TIME_OFFSET_US= -27800 score=0

Step 6: Interpolated cvs and corrected h5

In [18]:
#!/usr/bin/env python3
import h5py
import numpy as np

IN_H5 = (
    "/home/maurice/Documents/Master_Data/Periscope/"
    "Static_motion/dvSave_test_sync.h5"
)

ANGLE_CSV = (
    "/home/maurice/Documents/Master_Data/Periscope/"
    "Static_motion/angle_log.csv"
)

OUT_H5 = (
    "/home/maurice/Documents/Master_Data/Periscope/"
    "Static_motion/dvSave_corrected_interp_wide_vectorized.h5"
)

W, H = 640, 480

PROJECTION_W_PX = 11500
PAN_MARGIN_PX = 800

PAN_W = PROJECTION_W_PX + 2 * PAN_MARGIN_PX
PAN_H = 800

# This controls memory usage only.
# It is not a temporal window.
BATCH_SIZE = 500_000

DIRECTION = "right_to_left"

ROTATION_SIGN = 1.0
ROTATION_FACTOR = 1.0

TIME_OFFSET_US = -11100

DROP_OOB = True


def load_angle_csv(path):
    """
    Load angle measurements.

    The first CSV column is assumed to contain time in seconds.
    It is converted to integer microseconds.

    The second column is assumed to contain angle in degrees.
    """
    arr = np.loadtxt(path, delimiter=",", skiprows=1)

    if arr.ndim != 2 or arr.shape[1] < 2:
        raise ValueError(
            f"Expected at least two CSV columns, got shape {arr.shape}"
        )

    t = np.round(arr[:, 0] * 1_000_000).astype(np.int64)
    a = arr[:, 1].astype(np.float64)

    # Preserve the behavior of the original script:
    # discard initial samples until the first nonzero angle.
    nz = np.flatnonzero(a != 0.0)

    if nz.size:
        first = nz[0]
        t = t[first:]
        a = a[first:]

    if t.size == 0:
        raise ValueError("No angle samples remain after preprocessing.")

    # Ensure interpolation timestamps are monotonically ordered.
    order = np.argsort(t)
    t = t[order]
    a = a[order]

    # Remove duplicate timestamps because np.interp expects increasing x values.
    t, unique_idx = np.unique(t, return_index=True)
    a = a[unique_idx]

    t -= t[0]

    # Remove the discontinuity between 359 degrees and 0 degrees.
    a_unwrapped = np.rad2deg(
        np.unwrap(np.deg2rad(a))
    )

    return t, a_unwrapped


def interp_angle(t_query, t_angle, angle_unwrapped):
    """
    Interpolate an angle separately for every event timestamp.

    No temporal grouping or time-window approximation is performed.
    """
    tq = (
        np.asarray(t_query, dtype=np.float64)
        + float(TIME_OFFSET_US)
    )

    angle = np.interp(
        tq,
        t_angle.astype(np.float64),
        angle_unwrapped,
    )

    return np.mod(angle, 360.0)


def signed_angle_around_zero(angle_deg):
    """
    Convert angles to the range [-180, 180).
    """
    return ((angle_deg + 180.0) % 360.0) - 180.0


def angle_to_x(angle_deg):
    """
    Convert each interpolated angular position to a panorama x coordinate.
    """
    angle_signed = signed_angle_around_zero(angle_deg)

    if DIRECTION == "right_to_left":
        angle_signed = -angle_signed
    elif DIRECTION != "left_to_right":
        raise ValueError(
            "DIRECTION must be 'right_to_left' or 'left_to_right'."
        )

    return (
        PAN_W / 2.0
        + (angle_signed / 360.0) * PROJECTION_W_PX
    )


def correct_event_batch(
    events,
    t_sync,
    t_angle,
    angle_unwrapped,
):
    """
    Correct one memory-sized batch of events.

    Each event is corrected using the angle interpolated at that event's
    individual timestamp.

    Parameters
    ----------
    events:
        Event array with columns [x, y, timestamp].
    t_sync:
        Event timestamps relative to the common event-stream start.
    t_angle:
        Angle-log timestamps in microseconds.
    angle_unwrapped:
        Unwrapped angle-log measurements in degrees.

    Returns
    -------
    Corrected array with columns [panorama_x, panorama_y, timestamp].
    """
    if events.shape[0] == 0:
        return np.zeros((0, 3), dtype=np.int64)

    xy = events[:, :2].astype(np.float64)
    t = np.asarray(t_sync, dtype=np.int64)

    # One independently interpolated angle per event.
    angle_deg = interp_angle(
        t,
        t_angle,
        angle_unwrapped,
    )

    # Coordinates relative to the center of the original sensor.
    x_centered = xy[:, 0] - (W - 1) / 2.0
    y_centered = xy[:, 1] - (H - 1) / 2.0

    theta = (
        ROTATION_SIGN
        * ROTATION_FACTOR
        * np.deg2rad(angle_deg)
    )

    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    # Vectorized in-plane rotation.
    x_rotated = (
        x_centered * cos_theta
        - y_centered * sin_theta
    )

    y_rotated = (
        x_centered * sin_theta
        + y_centered * cos_theta
    )

    # Place each corrected event in the wide panorama.
    panorama_x = angle_to_x(angle_deg) + x_rotated
    panorama_y = PAN_H / 2.0 + y_rotated

    if DROP_OOB:
        valid = (
            (panorama_x >= 0.0)
            & (panorama_x < PAN_W)
            & (panorama_y >= 0.0)
            & (panorama_y < PAN_H)
        )

        panorama_x = panorama_x[valid]
        panorama_y = panorama_y[valid]
        t = t[valid]

    return np.column_stack([
        np.rint(panorama_x).astype(np.int64),
        np.rint(panorama_y).astype(np.int64),
        t,
    ])


def append_rows(dataset, rows):
    """
    Append corrected event rows to a resizable HDF5 dataset.
    """
    if rows.shape[0] == 0:
        return

    old_size = dataset.shape[0]
    new_size = old_size + rows.shape[0]

    dataset.resize((new_size, 3))
    dataset[old_size:new_size, :] = rows


def process_dataset(
    name,
    input_dataset,
    output_dataset,
    event_t0,
    t_angle,
    angle_unwrapped,
):
    """
    Process a complete polarity stream in memory-sized batches.

    Batch boundaries do not affect the correction. Every event receives its
    own interpolated angle based on its timestamp.
    """
    total_events = input_dataset.shape[0]

    print(f"Processing {name}: {total_events} events")

    for start in range(0, total_events, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total_events)

        events = np.asarray(
            input_dataset[start:end, :],
            dtype=np.int64,
        )

        # Use timestamps relative to the common pos/neg stream start.
        t_sync = events[:, 2] - event_t0

        corrected = correct_event_batch(
            events,
            t_sync,
            t_angle,
            angle_unwrapped,
        )

        append_rows(output_dataset, corrected)

        print(
            f"{name}: {end}/{total_events} "
            f"kept={output_dataset.shape[0]}"
        )


def main():
    t_angle, angle_unwrapped = load_angle_csv(ANGLE_CSV)

    print("PAN_W:", PAN_W)
    print("PAN_H:", PAN_H)
    print("PROJECTION_W_PX:", PROJECTION_W_PX)
    print("PAN_MARGIN_PX:", PAN_MARGIN_PX)
    print("BATCH_SIZE:", BATCH_SIZE)
    print("TIME_OFFSET_US:", TIME_OFFSET_US)
    print("DIRECTION:", DIRECTION)
    print("ROTATION_SIGN:", ROTATION_SIGN)
    print("ROTATION_FACTOR:", ROTATION_FACTOR)

    with (
        h5py.File(IN_H5, "r") as h5in,
        h5py.File(OUT_H5, "w") as h5out,
    ):
        dpos = h5in["pos"]
        dneg = h5in["neg"]

        if dpos.shape[0] == 0 and dneg.shape[0] == 0:
            raise ValueError("Both input event datasets are empty.")

        # Find one common timestamp origin without loading all timestamps.
        first_timestamps = []

        if dpos.shape[0] > 0:
            first_timestamps.append(int(dpos[0, 2]))

        if dneg.shape[0] > 0:
            first_timestamps.append(int(dneg[0, 2]))

        event_t0 = min(first_timestamps)

        pos_out = h5out.create_dataset(
            "pos",
            shape=(0, 3),
            maxshape=(None, 3),
            dtype=np.int64,
            chunks=True,
        )

        neg_out = h5out.create_dataset(
            "neg",
            shape=(0, 3),
            maxshape=(None, 3),
            dtype=np.int64,
            chunks=True,
        )

        h5out.attrs["coord_system"] = (
            "wide_panorama_vectorized_per_event"
        )
        h5out.attrs["PAN_W"] = PAN_W
        h5out.attrs["PAN_H"] = PAN_H
        h5out.attrs["PROJECTION_W_PX"] = PROJECTION_W_PX
        h5out.attrs["PAN_MARGIN_PX"] = PAN_MARGIN_PX
        h5out.attrs["BATCH_SIZE"] = BATCH_SIZE
        h5out.attrs["TIME_OFFSET_US"] = TIME_OFFSET_US
        h5out.attrs["DIRECTION"] = DIRECTION
        h5out.attrs["ROTATION_SIGN"] = ROTATION_SIGN
        h5out.attrs["ROTATION_FACTOR"] = ROTATION_FACTOR
        h5out.attrs["event_time_origin"] = event_t0
        h5out.attrs["correction_mode"] = (
            "continuous_per_event_interpolation"
        )

        process_dataset(
            name="pos",
            input_dataset=dpos,
            output_dataset=pos_out,
            event_t0=event_t0,
            t_angle=t_angle,
            angle_unwrapped=angle_unwrapped,
        )

        process_dataset(
            name="neg",
            input_dataset=dneg,
            output_dataset=neg_out,
            event_t0=event_t0,
            t_angle=t_angle,
            angle_unwrapped=angle_unwrapped,
        )

    print(
        "Saved vectorized per-event corrected panorama:",
        OUT_H5,
    )


if __name__ == "__main__":
    main()


PAN_W: 13100
PAN_H: 800
PROJECTION_W_PX: 11500
PAN_MARGIN_PX: 800
BATCH_SIZE: 500000
TIME_OFFSET_US: -11100
DIRECTION: right_to_left
ROTATION_SIGN: 1.0
ROTATION_FACTOR: 1.0
Processing pos: 55641805 events
pos: 500000/55641805 kept=500000
pos: 1000000/55641805 kept=1000000
pos: 1500000/55641805 kept=1500000
pos: 2000000/55641805 kept=2000000
pos: 2500000/55641805 kept=2500000
pos: 3000000/55641805 kept=3000000
pos: 3500000/55641805 kept=3500000
pos: 4000000/55641805 kept=4000000
pos: 4500000/55641805 kept=4500000
pos: 5000000/55641805 kept=5000000
pos: 5500000/55641805 kept=5500000
pos: 6000000/55641805 kept=6000000
pos: 6500000/55641805 kept=6500000
pos: 7000000/55641805 kept=7000000
pos: 7500000/55641805 kept=7500000
pos: 8000000/55641805 kept=8000000
pos: 8500000/55641805 kept=8500000
pos: 9000000/55641805 kept=9000000
pos: 9500000/55641805 kept=9500000
pos: 10000000/55641805 kept=10000000
pos: 10500000/55641805 kept=10500000
pos: 11000000/55641805 kept=11000000
pos: 11500000/5564180

For the second approach of motion segmentation, an edge extractor was needed. This is the edge detector based on ON- OFF polarity'

In [1]:
#!/usr/bin/env python3
import h5py
import numpy as np
from tqdm import tqdm

IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_corrected_interp_wide_vectorized.h5"
OUT_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_edges_onoff.h5"

EDGE_WINDOW_US = 5000   # try 1000, 5000, 10000


def normalize_uint8(img):
    m = img.max()
    if m == 0:
        return img.astype(np.uint8)
    return ((img.astype(np.float32) / m) * 255).astype(np.uint8)


def main():
    with h5py.File(IN_H5, "r") as h5in:
        pos = h5in["pos"]
        neg = h5in["neg"]

        # Your corrected H5 format:
        # pos / neg columns are [pano_x, pano_y, t]
        PAN_W = int(h5in.attrs["PAN_W"])
        PAN_H = int(h5in.attrs["PAN_H"])

        print("PAN_W:", PAN_W)
        print("PAN_H:", PAN_H)

        t_pos = pos[:, 2].astype(np.int64)
        t_neg = neg[:, 2].astype(np.int64)

        t0 = min(t_pos[0], t_neg[0])
        t1 = max(t_pos[-1], t_neg[-1])

        num_frames = int(np.ceil((t1 - t0) / EDGE_WINDOW_US))

        with h5py.File(OUT_H5, "w") as h5out:
            edges_ds = h5out.create_dataset(
                "edges",
                shape=(num_frames, PAN_H, PAN_W),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, PAN_H, PAN_W),
            )

            on_ds = h5out.create_dataset(
                "on",
                shape=(num_frames, PAN_H, PAN_W),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, PAN_H, PAN_W),
            )

            off_ds = h5out.create_dataset(
                "off",
                shape=(num_frames, PAN_H, PAN_W),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, PAN_H, PAN_W),
            )

            times_ds = h5out.create_dataset(
                "t",
                shape=(num_frames,),
                dtype=np.int64,
            )

            p0 = 0
            n0 = 0

            for i in tqdm(range(num_frames)):
                win_start = t0 + i * EDGE_WINDOW_US
                win_end = win_start + EDGE_WINDOW_US

                p1 = np.searchsorted(t_pos, win_end, side="left")
                n1 = np.searchsorted(t_neg, win_end, side="left")

                pos_win = pos[p0:p1]
                neg_win = neg[n0:n1]

                on_img = np.zeros((PAN_H, PAN_W), dtype=np.uint16)
                off_img = np.zeros((PAN_H, PAN_W), dtype=np.uint16)

                xp = pos_win[:, 0]
                yp = pos_win[:, 1]

                xn = neg_win[:, 0]
                yn = neg_win[:, 1]

                valid_p = (
                    (xp >= 0) & (xp < PAN_W) &
                    (yp >= 0) & (yp < PAN_H)
                )

                valid_n = (
                    (xn >= 0) & (xn < PAN_W) &
                    (yn >= 0) & (yn < PAN_H)
                )

                np.add.at(on_img, (yp[valid_p], xp[valid_p]), 1)
                np.add.at(off_img, (yn[valid_n], xn[valid_n]), 1)

                # Retina-style ON/OFF opponency
                edge = np.abs(
                    on_img.astype(np.int16) -
                    off_img.astype(np.int16)
                )

                edges_ds[i] = normalize_uint8(edge)
                on_ds[i] = normalize_uint8(on_img)
                off_ds[i] = normalize_uint8(off_img)
                times_ds[i] = win_start

                p0 = p1
                n0 = n1

            h5out.attrs["PAN_W"] = PAN_W
            h5out.attrs["PAN_H"] = PAN_H
            h5out.attrs["EDGE_WINDOW_US"] = EDGE_WINDOW_US
            h5out.attrs["method"] = "edges = abs(pos_count - neg_count)"
            h5out.attrs["input_format"] = "[pano_x, pano_y, t]"

    print("Saved:", OUT_H5)


if __name__ == "__main__":
    main()

PAN_W: 13100
PAN_H: 800


100%|██████████| 3067/3067 [09:45<00:00,  5.24it/s]

Saved: /home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_edges_onoff.h5


Visualizer

In [ ]:
#!/usr/bin/env python3
import h5py
import numpy as np
import cv2

IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/dvSave_corrected_interp_wide_vectorized.h5"
OUT_MP4 = "/home/maurice/Documents/Master_Data/Periscope/Static_motion/corrected_full_accumulator_wide_comb.mp4"

PAN_W, PAN_H = 13000, 800
OUT_W, OUT_H = 1280, 400

FPS = 30
WINDOW_US = 33_000

ACC_DECAY = 0.95
ACC_GAIN = 1

CLIP_ACC = 10.0
CLIP_WIN = 1


def counts_image_from_xy(xy, h, w):
    if xy.size == 0:
        return np.zeros((h, w), dtype=np.float32)

    x = xy[:, 0].astype(np.int64)
    y = xy[:, 1].astype(np.int64)

    valid = (x >= 0) & (x < w) & (y >= 0) & (y < h)
    x = x[valid]
    y = y[valid]

    if x.size == 0:
        return np.zeros((h, w), dtype=np.float32)

    idx = y * w + x
    return np.bincount(idx, minlength=h * w).reshape(h, w).astype(np.float32)


def make_bgr(acc_pos, acc_neg, win_pos, win_neg):
    bgr = np.zeros((PAN_H, PAN_W, 3), dtype=np.uint8)

    acc_pos8 = (np.clip(acc_pos, 0, CLIP_ACC) * 255.0 / CLIP_ACC).astype(np.uint8)
    acc_neg8 = (np.clip(acc_neg, 0, CLIP_ACC) * 255.0 / CLIP_ACC).astype(np.uint8)

    win_pos8 = (np.clip(win_pos, 0, CLIP_WIN) * 255.0 / CLIP_WIN).astype(np.uint8)
    win_neg8 = (np.clip(win_neg, 0, CLIP_WIN) * 255.0 / CLIP_WIN).astype(np.uint8)

    bgr[..., 2] = acc_pos8
    bgr[..., 0] = acc_neg8

    bgr[..., 1] = np.maximum(bgr[..., 1], win_pos8)
    bgr[..., 0] = np.maximum(bgr[..., 0], win_neg8)
    bgr[..., 1] = np.maximum(bgr[..., 1], win_neg8 // 2)

    return bgr


def main():
    with h5py.File(IN_H5, "r") as h5:
        dpos = h5["pos"]
        dneg = h5["neg"]

        tpos = dpos[:, 2].astype(np.int64)
        tneg = dneg[:, 2].astype(np.int64)

        t_min = min(tpos[0], tneg[0])
        t_max = max(tpos[-1], tneg[-1])

        times = np.arange(t_min, t_max, WINDOW_US, dtype=np.int64)

        acc_pos = np.zeros((PAN_H, PAN_W), dtype=np.float32)
        acc_neg = np.zeros((PAN_H, PAN_W), dtype=np.float32)

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        vw = cv2.VideoWriter(OUT_MP4, fourcc, FPS, (OUT_W, OUT_H), True)

        for k, t_start in enumerate(times):
            t_end = t_start + WINDOW_US

            pi0 = np.searchsorted(tpos, t_start, side="left")
            pi1 = np.searchsorted(tpos, t_end, side="left")

            ni0 = np.searchsorted(tneg, t_start, side="left")
            ni1 = np.searchsorted(tneg, t_end, side="left")

            pos_xy = np.asarray(dpos[pi0:pi1, :2], dtype=np.int64)
            neg_xy = np.asarray(dneg[ni0:ni1, :2], dtype=np.int64)

            win_pos = counts_image_from_xy(pos_xy, PAN_H, PAN_W)
            win_neg = counts_image_from_xy(neg_xy, PAN_H, PAN_W)

            acc_pos *= ACC_DECAY
            acc_neg *= ACC_DECAY

            acc_pos += win_pos * ACC_GAIN
            acc_neg += win_neg * ACC_GAIN

            frame = make_bgr(acc_pos, acc_neg, win_pos, win_neg)

            cv2.putText(
                frame,
                f"k={k}/{len(times)} win={WINDOW_US}us decay={ACC_DECAY} gain={ACC_GAIN}",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            out = cv2.resize(frame, (OUT_W, OUT_H), interpolation=cv2.INTER_AREA)
            vw.write(out)

            if k % 500 == 0:
                print(
                    f"k={k}/{len(times)} "
                    f"pos={pos_xy.shape[0]} "
                    f"neg={neg_xy.shape[0]} "
                    f"acc_max=({acc_pos.max():.1f},{acc_neg.max():.1f})"
                )

        vw.release()

    print("Saved:", OUT_MP4)


if __name__ == "__main__":
    main()

k=0/465 pos=222853 neg=359673 acc_max=(28.0,33.0)
Saved: /home/maurice/Documents/Master_Data/Periscope/Static_motion/corrected_full_accumulator_wide_comb.mp4


: 